In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

In [2]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

RAW_DIR = PROJECT_ROOT / "DATA" / "RAW"
PROCESSED_DIR = PROJECT_ROOT / "DATA" / "PROCESSED"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project Root:", PROJECT_ROOT)
print("Raw Directory:", RAW_DIR)
print("Processed Directory:", PROCESSED_DIR)

Project Root: C:\Users\sitar\Downloads\ASSIGNMENT_CODE\CARS
Raw Directory: C:\Users\sitar\Downloads\ASSIGNMENT_CODE\CARS\DATA\RAW
Processed Directory: C:\Users\sitar\Downloads\ASSIGNMENT_CODE\CARS\DATA\PROCESSED


In [3]:
csv_files = list(RAW_DIR.glob("*.csv"))

print("CSV files found:")

for file in csv_files:
    print(" -", file.name)

CSV files found:
 - cars.csv


In [4]:
if not csv_files:
    raise FileNotFoundError(
        f"No CSV file found in {RAW_DIR}. "
        "Please place the downloaded CSV inside data/raw."
    )

RAW_FILE = csv_files[0]

print("Using dataset:")
print(RAW_FILE)

Using dataset:
C:\Users\sitar\Downloads\ASSIGNMENT_CODE\CARS\DATA\RAW\cars.csv


In [5]:
cars = pd.read_csv(RAW_FILE)

print("Dataset loaded successfully.")
print("Shape:", cars.shape)

cars.head()

Dataset loaded successfully.
Shape: (1725, 9)


,Id,year,brand,full_model_name,model_name,price,distance_travelled(kms),fuel_type,city
0,0,2016,Honda,Honda Brio S MT,Brio,425000.0,9680.0,Petrol,Mumbai
1,1,2012,Nissan,Nissan Sunny XV Diesel,Sunny,325000.0,119120.0,Diesel,Mumbai
2,2,2017,Toyota,Toyota Fortuner 2.8 4x2 MT [2016-2020],Fortuner,2650000.0,64593.0,Diesel,Thane
3,3,2017,Mercedes-Benz,Mercedes-Benz E-Class E 220d Expression [2019-...,E-Class,4195000.0,25000.0,Diesel,Mumbai
4,4,2012,Hyundai,Hyundai Verna Fluidic 1.6 CRDi SX,Verna,475000.0,23800.0,Diesel,Mumbai


In [6]:
print("Number of rows:", cars.shape[0])
print("Number of columns:", cars.shape[1])

Number of rows: 1725
Number of columns: 9


In [7]:
print("Columns:")
for i, column in enumerate(cars.columns, start=1):
    print(f"{i}. {column}")
    
cars.info()

Columns:
1. Id
2. year
3. brand
4. full_model_name
5. model_name
6. price
7. distance_travelled(kms)
8. fuel_type
9. city
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1725 entries, 0 to 1724
Data columns (total 9 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Id                       1725 non-null   int64  
 1   year                     1725 non-null   int64  
 2   brand                    1725 non-null   object 
 3   full_model_name          1725 non-null   object 
 4   model_name               1725 non-null   object 
 5   price                    1725 non-null   float64
 6   distance_travelled(kms)  1725 non-null   float64
 7   fuel_type                1725 non-null   object 
 8   city                     1725 non-null   object 
dtypes: float64(2), int64(2), object(5)
memory usage: 121.4+ KB


In [8]:
missing_values = cars.isnull().sum()

missing_table = pd.DataFrame({
    "column": missing_values.index,
    "missing_count": missing_values.values,
    "missing_percentage": (
        missing_values.values / len(cars) * 100
    ).round(2)
})

missing_table = missing_table.sort_values(
    "missing_count",
    ascending=False
)

missing_table

,column,missing_count,missing_percentage
0,Id,0,0.0
1,year,0,0.0
2,brand,0,0.0
3,full_model_name,0,0.0
4,model_name,0,0.0
5,price,0,0.0
6,distance_travelled(kms),0,0.0
7,fuel_type,0,0.0
8,city,0,0.0


In [9]:
duplicate_count = cars.duplicated().sum()

print("Duplicate rows:", duplicate_count)

if duplicate_count > 0:
    cars = cars.drop_duplicates().reset_index(drop=True)

print("Shape after removing duplicates:", cars.shape)

Duplicate rows: 0
Shape after removing duplicates: (1725, 9)


In [10]:
cars.columns = (
    cars.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

print(cars.columns.tolist())

['id', 'year', 'brand', 'full_model_name', 'model_name', 'price', 'distance_travelled(kms)', 'fuel_type', 'city']


In [11]:
dtype_table = pd.DataFrame({
    "column": cars.columns,
    "dtype": cars.dtypes.astype(str).values
})

dtype_table

,column,dtype
0,id,int64
1,year,int64
2,brand,object
3,full_model_name,object
4,model_name,object
5,price,float64
6,distance_travelled(kms),float64
7,fuel_type,object
8,city,object


In [12]:
categorical_columns = cars.select_dtypes(
    include=["object", "category"]
).columns

print("Categorical columns:")

for column in categorical_columns:
    print(f"\n--- {column} ---")
    print(cars[column].value_counts(dropna=False).head(20))

Categorical columns:

--- brand ---
brand
Hyundai          297
Maruti Suzuki    275
Honda            153
Mercedes-Benz    131
Toyota           117
BMW              111
Audi              98
Mahindra          93
Ford              78
Volkswagen        70
Renault           51
Skoda             45
Tata              44
Jaguar            30
Land Rover        28
Volvo             17
Nissan            13
Kia               12
MINI              10
Jeep              10
Name: count, dtype: int64

--- full_model_name ---
full_model_name
Ford EcoSport Titanium 1.5L TDCi                 12
Toyota Innova Crysta 2.4 VX 8 STR [2016-2020]    11
Honda City 1.5 S MT                              11
Audi Q7 3.0 TDI quattro Premium Plus             11
Volkswagen Polo GT TSI                           11
Toyota Innova Crysta 2.4 VX 7 STR [2016-2020]     9
Renault Kwid 1.0 RXT AMT Opt [2016-2019]          9
Honda WR-V VX MT Diesel                           9
Mercedes-Benz C-Class C 220 CDI Avantgarde        9
For

In [13]:
numeric_columns = cars.select_dtypes(
    include=np.number
).columns

print("Numerical columns:")
print(list(numeric_columns))

cars[numeric_columns].describe().T

Numerical columns:
['id', 'year', 'price', 'distance_travelled(kms)']


,count,mean,std,min,25%,50%,75%,max
id,1725.0,8.620000e+02,4.981089e+02,0.0,431.0,862.0,1293.0,1724.0
year,1725.0,2.015391e+03,3.207504e+00,1990.0,2013.0,2016.0,2018.0,2021.0
price,1725.0,1.494837e+06,1.671658e+06,62500.0,545000.0,875000.0,1825000.0,14700000.0
distance_travelled(kms),1725.0,5.384826e+04,4.472554e+04,350.0,29000.0,49000.0,70500.0,790000.0


In [14]:
for column in cars.select_dtypes(include="object").columns:
    cars[column] = cars[column].apply(
        lambda x: x.strip() if isinstance(x, str) else x
    )

print("String values cleaned.")

String values cleaned.


In [15]:
missing_tokens = [
    "",
    " ",
    "NA",
    "N/A",
    "na",
    "n/a",
    "null",
    "NULL",
    "None",
    "none",
    "-"
]

cars = cars.replace(missing_tokens, np.nan)

print("Missing-value representations standardized.")

Missing-value representations standardized.


In [16]:
possible_numeric_rules = {
    "year": lambda x: x > 0,
    "selling_price": lambda x: x > 0,
    "price": lambda x: x > 0,
    "km_driven": lambda x: x >= 0,
    "kilometers_driven": lambda x: x >= 0,
}

for column, rule in possible_numeric_rules.items():
    if column in cars.columns:
        before = len(cars)
        cars = cars[cars[column].isna() | rule(cars[column])]
        after = len(cars)
        print(f"{column}: removed {before - after} invalid rows")

year: removed 0 invalid rows
price: removed 0 invalid rows


In [17]:
numeric_columns = cars.select_dtypes(include=np.number).columns
categorical_columns = cars.select_dtypes(
    include=["object", "category"]
).columns

for column in numeric_columns:
    if cars[column].isna().sum() > 0:
        cars[column] = cars[column].fillna(cars[column].median())

for column in categorical_columns:
    if cars[column].isna().sum() > 0:
        mode = cars[column].mode()

        if len(mode) > 0:
            cars[column] = cars[column].fillna(mode.iloc[0])
        else:
            cars[column] = cars[column].fillna("Unknown")

print("Missing values handled.")

Missing values handled.


In [18]:
print("Remaining missing values:")

remaining_missing = cars.isnull().sum()

print(
    remaining_missing[
        remaining_missing > 0
    ]
)

Remaining missing values:
Series([], dtype: int64)


In [19]:
print("Final shape:", cars.shape)

display(cars.head())

cars.describe(include="all").T

Final shape: (1725, 9)


,id,year,brand,full_model_name,model_name,price,distance_travelled(kms),fuel_type,city
0,0,2016,Honda,Honda Brio S MT,Brio,425000.0,9680.0,Petrol,Mumbai
1,1,2012,Nissan,Nissan Sunny XV Diesel,Sunny,325000.0,119120.0,Diesel,Mumbai
2,2,2017,Toyota,Toyota Fortuner 2.8 4x2 MT [2016-2020],Fortuner,2650000.0,64593.0,Diesel,Thane
3,3,2017,Mercedes-Benz,Mercedes-Benz E-Class E 220d Expression [2019-...,E-Class,4195000.0,25000.0,Diesel,Mumbai
4,4,2012,Hyundai,Hyundai Verna Fluidic 1.6 CRDi SX,Verna,475000.0,23800.0,Diesel,Mumbai


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
id,1725.0,NaN,NaN,NaN,862.0,498.108924,0.0,431.0,862.0,1293.0,1724.0
year,1725.0,NaN,NaN,NaN,2015.390725,3.207504,1990.0,2013.0,2016.0,2018.0,2021.0
brand,1725,31,Hyundai,297,NaN,NaN,NaN,NaN,NaN,NaN,NaN
full_model_name,1725,750,Ford EcoSport Titanium 1.5L TDCi,12,NaN,NaN,NaN,NaN,NaN,NaN,NaN
model_name,1725,169,Creta,71,NaN,NaN,NaN,NaN,NaN,NaN,NaN
price,1725.0,NaN,NaN,NaN,1494837.391304,1671657.727986,62500.0,545000.0,875000.0,1825000.0,14700000.0
distance_travelled(kms),1725.0,NaN,NaN,NaN,53848.256232,44725.541963,350.0,29000.0,49000.0,70500.0,790000.0
fuel_type,1725,5,Diesel,922,NaN,NaN,NaN,NaN,NaN,NaN,NaN
city,1725,15,Chennai,493,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
PROCESSED_FILE = PROCESSED_DIR / "cars_clean.csv"

cars.to_csv(
    PROCESSED_FILE,
    index=False
)

print("Clean dataset saved to:")
print(PROCESSED_FILE)

Clean dataset saved to:
C:\Users\sitar\Downloads\ASSIGNMENT_CODE\CARS\DATA\PROCESSED\cars_clean.csv


In [21]:
print("Final dataset shape:", cars.shape)
print("File exists:", PROCESSED_FILE.exists())

Final dataset shape: (1725, 9)
File exists: True
